# Half-Truth LaTeX Table Generator

Generates LaTeX tables for:
1. **Summary Table**: Entity vs Relation accuracy with micro-averaged gaps
2. **Per-Condition Table**: Detailed accuracy per condition
3. **Full-Truth Table**: Whether Full-Truth > Half-Truth

In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
import os

# =============================================================================
# CONFIGURATION
# =============================================================================

RESULTS_BASE_DIR = "../../results_new"

# Auto-discover all models from results_new directory
def discover_models(base_dir):
    """Auto-discover all half_truth model folders."""
    models = {}
    base_path = Path(base_dir)
    
    # Model name mapping: folder suffix -> display name
    NAME_MAP = {
        "00_baseline_openai_vitb32": "CLIP",
        "laclip_cc12m_vitb32": "LA-CLIP",
        "negclip_coco_vitb32": "NegCLIP",
        "tripletclip_cc12m_vitb12": "TripletCLIP",
        "cyclip_vitb32": "CyCLIP",
        "ce_clip_vitb32": "CE-CLIP",
        "con_clip_vitb32": "CON-CLIP",
        "dac_vitb32_llm": "DAC-LLM",
        "dac_vitb32_sam": "DAC-SAM",
        "degla_vitb32": "DEGLA",
        "clic_cogvlm_vitb32_laion": "CLIC-LAION",
        "clic_cogvlm_vitb32_pixpr_redcaps": "CLIC-RedCaps",
        "clove_vitb32_laioncoco": "CLoVe",
        "fsc_clip_cc3m_vitb32": "FSC-CLIP-CC3M",
        "fsc_clip_coco_vitb32": "FSC-CLIP-COCO",
        "fsc_clip_laioncoco_vitb32": "FSC-CLIP-LAION",
        "labclip_vitb32": "LabCLIP",
        "readclip_vitb32": "ReadCLIP",
        "siglip_vitb16": "SigLIP",
        "siglip2_vitb16": "SigLIP2",
        "tsvlc_vitb32": "TSVLC",
        "cs_clip_negclip_vitb32": "CS-CLIP",
    }
    
    for folder in sorted(base_path.iterdir()):
        if folder.is_dir() and folder.name.startswith("half_truth_coco_"):
            suffix = folder.name.replace("half_truth_coco_", "")
            display_name = NAME_MAP.get(suffix, suffix)
            models[display_name] = folder.name
        if folder.is_dir() and folder.name.startswith("half_truth_laion_"):
            suffix = folder.name.replace("half_truth_laion_", "")
            display_name = NAME_MAP.get(suffix, suffix)
            models[display_name] = folder.name
    return models

MODELS = discover_models(RESULTS_BASE_DIR)

# Filter models
# FILTER = [
#     "CLIP",
#     "NegCLIP",
#     "SigLIP",
#     "SigLIP2",
# ]
# MODELS = {name: folder for name, folder in MODELS.items() if name in FILTER}
print(f"Discovered {len(MODELS)} models:")
for name, folder in MODELS.items():
    print(f"  {name}: {folder}")

# Condition mapping: internal name -> (display_name, category)
# Excluding relation_negation
CONDITION_MAP = {
    # Entity foils
    'component_easy': ('+Obj', 'Entity'),
    'component_hard': ('+Attr', 'Entity'),
    'component_random': ('+Rand', 'Entity'),
    # Relation foils (excluding relation_negation)
    'attribute_wrong': ('Attr', 'Relation'),
    'object_wrong': ('Obj', 'Relation'),
    'relation_antonym': ('Ant', 'Relation'),
    'relation_swap': ('Swap', 'Relation'),
    'subject_wrong': ('Subj', 'Relation'),
}

ENTITY_CONDITIONS = ['component_easy', 'component_hard', 'component_random']
RELATION_CONDITIONS = ['attribute_wrong', 'object_wrong', 'relation_antonym', 
                       'relation_swap', 'subject_wrong']

# =============================================================================
# LOADING FUNCTIONS
# =============================================================================

def load_model_results(base_dir, model_folder):
    """Load results for a single model."""
    model_path = Path(base_dir) / model_folder
    results_file = model_path / 'results.json'
    
    if results_file.exists():
        with open(results_file, 'r') as f:
            data = json.load(f)
            if 'results' in data:
                return pd.DataFrame(data['results'])
            else:
                return pd.DataFrame(data)
    return None


def load_all_models(base_dir, models):
    """Load results for all models."""
    all_results = {}
    
    for display_name, folder_name in models.items():
        try:
            df = load_model_results(base_dir, folder_name)
            if df is not None:
                df['model'] = display_name
                all_results[display_name] = df
                print(f"  ✓ {display_name}: {len(df)} samples")
            else:
                print(f"  ⚠ {display_name}: No results found")
        except Exception as e:
            print(f"  ✗ {display_name}: Error - {e}")
    
    return all_results


# Load data
print("="*60)
print("LOADING HALF-TRUTH RESULTS")
print("="*60)

all_model_results = load_all_models(RESULTS_BASE_DIR, MODELS)
print(f"\n✓ Loaded {len(all_model_results)} models total")

Discovered 22 models:
  CLIP: half_truth_coco_00_baseline_openai_vitb32
  CE-CLIP: half_truth_coco_ce_clip_vitb32
  CLIC-LAION: half_truth_coco_clic_cogvlm_vitb32_laion
  CLIC-RedCaps: half_truth_coco_clic_cogvlm_vitb32_pixpr_redcaps
  CLoVe: half_truth_coco_clove_vitb32_laioncoco
  CON-CLIP: half_truth_coco_con_clip_vitb32
  CS-CLIP: half_truth_coco_cs_clip_negclip_vitb32
  CyCLIP: half_truth_coco_cyclip_vitb32
  DAC-LLM: half_truth_coco_dac_vitb32_llm
  DAC-SAM: half_truth_coco_dac_vitb32_sam
  DEGLA: half_truth_coco_degla_vitb32
  FSC-CLIP-CC3M: half_truth_coco_fsc_clip_cc3m_vitb32
  FSC-CLIP-COCO: half_truth_coco_fsc_clip_coco_vitb32
  FSC-CLIP-LAION: half_truth_coco_fsc_clip_laioncoco_vitb32
  LabCLIP: half_truth_coco_labclip_vitb32
  LA-CLIP: half_truth_coco_laclip_cc12m_vitb32
  NegCLIP: half_truth_coco_negclip_coco_vitb32
  ReadCLIP: half_truth_coco_readclip_vitb32
  SigLIP2: half_truth_coco_siglip2_vitb16
  SigLIP: half_truth_coco_siglip_vitb16
  TripletCLIP: half_truth_coco_t

In [2]:
# =============================================================================
# COMPUTE METRICS (MICRO-AVERAGED)
# =============================================================================

def filter_negation(df):
    """Filter out relation_negation condition."""
    return df[df['condition'] != 'relation_negation'].copy()


def compute_micro_avg_accuracy(df, conditions):
    """
    Compute micro-averaged accuracy over a set of conditions.
    
    HT-Acc = percentage where score_short_correct > score_long_incorrect
    (Ties go to half-truth, i.e., st_wins = 0 when scores are equal)
    
    Args:
        df: DataFrame with results
        conditions: List of condition names to include
    
    Returns:
        accuracy as percentage, sample count
    """
    subset = df[df['condition'].isin(conditions)]
    if len(subset) == 0:
        return np.nan, 0
    
    # Re-compute from raw scores: ST wins only if strictly greater
    st_wins = (subset['score_short_correct'] > subset['score_long_incorrect']).astype(float)
    acc = st_wins.mean() * 100
    
    return acc, len(subset)


def compute_per_condition_accuracy(df, condition_map):
    """
    Compute accuracy per condition.
    
    HT-Acc = percentage where score_short_correct > score_long_incorrect
    (Ties go to half-truth)
    """
    records = []
    for condition, (display_name, category) in condition_map.items():
        subset = df[df['condition'] == condition]
        if len(subset) == 0:
            continue
        
        # Re-compute from raw scores: ST wins only if strictly greater
        st_wins = (subset['score_short_correct'] > subset['score_long_incorrect']).astype(float)
        acc = st_wins.mean() * 100
        
        records.append({
            'condition': condition,
            'display_name': display_name,
            'category': category,
            'accuracy': acc,
            'n': len(subset),
        })
    return pd.DataFrame(records)


# Compute metrics for all models
print("\n" + "="*60)
print("COMPUTING MICRO-AVERAGED METRICS")
print("="*60)

summary_records = []
per_condition_records = []

for model_name, df in all_model_results.items():
    # Filter negation
    df_filtered = filter_negation(df)
    
    # Entity accuracy (micro-averaged)
    ent_acc, ent_n = compute_micro_avg_accuracy(df_filtered, ENTITY_CONDITIONS)
    
    # Relation accuracy (micro-averaged)
    rel_acc, rel_n = compute_micro_avg_accuracy(df_filtered, RELATION_CONDITIONS)
    
    # Overall accuracy (micro-averaged)
    all_conditions = ENTITY_CONDITIONS + RELATION_CONDITIONS
    overall_acc, overall_n = compute_micro_avg_accuracy(df_filtered, all_conditions)
    
    # Gap = Entity - Relation
    gap = ent_acc - rel_acc if not np.isnan(ent_acc) and not np.isnan(rel_acc) else np.nan
    
    summary_records.append({
        'Model': model_name,
        'Entity': ent_acc,
        'Entity_n': ent_n,
        'Relation': rel_acc,
        'Relation_n': rel_n,
        'Gap': gap,
        'Overall': overall_acc,
        'Overall_n': overall_n,
    })
    
    # Per-condition accuracy
    cond_df = compute_per_condition_accuracy(df_filtered, CONDITION_MAP)
    cond_df['model'] = model_name
    per_condition_records.append(cond_df)

summary_df = pd.DataFrame(summary_records).set_index('Model')
per_condition_df = pd.concat(per_condition_records, ignore_index=True)

print("\nSummary (Micro-Averaged HT-Acc %):")
print(summary_df[['Entity', 'Relation', 'Gap', 'Overall']].round(1).to_string())


COMPUTING MICRO-AVERAGED METRICS



Summary (Micro-Averaged HT-Acc %):
                Entity  Relation   Gap  Overall
Model                                          
CLIP              52.9      32.9  20.0     40.6
CE-CLIP           59.1      42.9  16.1     49.1
CLIC-LAION        61.5      29.1  32.4     41.5
CLIC-RedCaps      57.5      30.4  27.1     40.8
CLoVe             57.9      29.6  28.4     40.4
CON-CLIP          76.0      48.8  27.2     59.2
CS-CLIP           75.4      65.5  10.0     69.3
DAC-LLM           61.1      37.1  24.0     46.3
DAC-SAM           58.6      37.0  21.6     45.3
DEGLA             68.3      38.2  30.1     49.7
FSC-CLIP-CC3M     77.7      54.1  23.6     63.1
FSC-CLIP-COCO     74.9      47.9  27.0     58.2
FSC-CLIP-LAION    70.2      45.9  24.3     55.2
LabCLIP           64.2      39.2  25.1     48.8
LA-CLIP           66.7      47.8  18.8     55.1
NegCLIP           69.8      48.3  21.5     56.5
ReadCLIP          69.4      51.0  18.4     58.0
SigLIP2           69.9      45.2  24.7     54.6
SigL

In [3]:
# =============================================================================
# TABLE 1: SUMMARY TABLE (Entity vs Relation with Gap)
# =============================================================================

def generate_summary_latex_table(summary_df, model_order=None):
    """Generate LaTeX table for Entity vs Relation summary with Gap."""
    
    if model_order is None:
        # Sort by Overall accuracy descending
        model_order = summary_df.sort_values('Overall', ascending=False).index.tolist()
    else:
        model_order = [m for m in model_order if m in summary_df.index]
    
    # Find best per column (highest accuracy, largest positive gap)
    best_entity = summary_df['Entity'].idxmax()
    best_relation = summary_df['Relation'].idxmax()
    best_overall = summary_df['Overall'].idxmax()
    # For gap, we might want smallest gap (closest to 0) or largest positive
    # Let's highlight smallest absolute gap
    best_gap = summary_df['Gap'].abs().idxmin()
    
    def fmt_acc(val, model, best_model):
        s = f"{val:.1f}"
        if model == best_model:
            s = f"\\textbf{{{s}}}"
        return s
    
    def fmt_gap(val, model, best_model):
        if val > 0:
            s = f"+{val:.1f}"
        else:
            s = f"{val:.1f}"
        if model == best_model:
            s = f"\\textbf{{{s}}}"
        return s
    
    lines = []
    lines.append(r"\begin{tabular}{@{}lcccc@{}}")
    lines.append(r"\toprule")
    lines.append(r"\textbf{Model} & \textbf{Entity} & \textbf{Relation} & \textbf{Gap} & \textbf{Overall} \\")
    lines.append(r"\midrule")
    
    for model in model_order:
        row = model.replace("_", r"\_")
        row += f" & {fmt_acc(summary_df.loc[model, 'Entity'], model, best_entity)}"
        row += f" & {fmt_acc(summary_df.loc[model, 'Relation'], model, best_relation)}"
        row += f" & {fmt_gap(summary_df.loc[model, 'Gap'], model, best_gap)}"
        row += f" & {fmt_acc(summary_df.loc[model, 'Overall'], model, best_overall)} \\\\"
        lines.append(row)
    
    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    
    return "\n".join(lines)


# Generate summary table
summary_latex = generate_summary_latex_table(summary_df)

print("\n" + "="*80)
print("TABLE 1: SUMMARY (Entity vs Relation, Micro-Averaged)")
print("="*80)
print(summary_latex)


TABLE 1: SUMMARY (Entity vs Relation, Micro-Averaged)
\begin{tabular}{@{}lcccc@{}}
\toprule
\textbf{Model} & \textbf{Entity} & \textbf{Relation} & \textbf{Gap} & \textbf{Overall} \\
\midrule
CS-CLIP & 75.4 & \textbf{65.5} & \textbf{+10.0} & \textbf{69.3} \\
FSC-CLIP-CC3M & \textbf{77.7} & 54.1 & +23.6 & 63.1 \\
CON-CLIP & 76.0 & 48.8 & +27.2 & 59.2 \\
FSC-CLIP-COCO & 74.9 & 47.9 & +27.0 & 58.2 \\
ReadCLIP & 69.4 & 51.0 & +18.4 & 58.0 \\
NegCLIP & 69.8 & 48.3 & +21.5 & 56.5 \\
FSC-CLIP-LAION & 70.2 & 45.9 & +24.3 & 55.2 \\
LA-CLIP & 66.7 & 47.8 & +18.8 & 55.1 \\
SigLIP2 & 69.9 & 45.2 & +24.7 & 54.6 \\
DEGLA & 68.3 & 38.2 & +30.1 & 49.7 \\
CE-CLIP & 59.1 & 42.9 & +16.1 & 49.1 \\
LabCLIP & 64.2 & 39.2 & +25.1 & 48.8 \\
DAC-LLM & 61.1 & 37.1 & +24.0 & 46.3 \\
TripletCLIP & 60.0 & 37.2 & +22.9 & 45.9 \\
TSVLC & 59.6 & 37.3 & +22.3 & 45.8 \\
SigLIP & 56.9 & 38.8 & +18.1 & 45.7 \\
DAC-SAM & 58.6 & 37.0 & +21.6 & 45.3 \\
CLIC-LAION & 61.5 & 29.1 & +32.4 & 41.5 \\
CLIC-RedCaps & 57.5 & 30.4 & 

In [4]:
# =============================================================================
# TABLE 2: PER-CONDITION TABLE
# =============================================================================

def generate_per_condition_latex_table(per_condition_df, condition_map, 
                                        entity_conditions, relation_conditions,
                                        model_order=None):
    """Generate LaTeX table with per-condition accuracy."""
    
    # Pivot to model x condition
    pivot = per_condition_df.pivot(index='model', columns='condition', values='accuracy')
    
    if model_order is None:
        model_order = sorted(pivot.index.tolist())
    else:
        model_order = [m for m in model_order if m in pivot.index]
    
    # Filter to existing conditions
    entity_cols = [c for c in entity_conditions if c in pivot.columns]
    relation_cols = [c for c in relation_conditions if c in pivot.columns]
    all_cols = entity_cols + relation_cols
    
    # Find best per column
    best_per_col = {col: pivot[col].idxmax() for col in all_cols}
    
    def fmt(val, model, col):
        s = f"{val:.1f}"
        if best_per_col.get(col) == model:
            s = f"\\textbf{{{s}}}"
        return s
    
    # Headers
    entity_headers = [condition_map[c][0] for c in entity_cols]
    relation_headers = [condition_map[c][0] for c in relation_cols]
    
    n_entity = len(entity_cols)
    n_relation = len(relation_cols)
    
    lines = []
    lines.append(r"\begin{tabular}{@{}l" + "c" * n_entity + "|" + "c" * n_relation + "@{}}")
    lines.append(r"\toprule")
    
    # Group headers
    lines.append(f"& \\multicolumn{{{n_entity}}}{{c|}}{{\\textbf{{Entity}}}} & \\multicolumn{{{n_relation}}}{{c}}{{\\textbf{{Relation}}}} \\\\")
    lines.append(f"\\cmidrule(r){{2-{1+n_entity}}}\\cmidrule(l){{{2+n_entity}-{1+n_entity+n_relation}}}")
    
    # Column headers
    header_row = "\\textbf{Model}"
    for h in entity_headers:
        header_row += f" & {h}"
    for h in relation_headers:
        header_row += f" & {h}"
    header_row += " \\\\"
    lines.append(header_row)
    lines.append(r"\midrule")
    
    # Data rows
    for model in model_order:
        row = model.replace("_", r"\_")
        for col in entity_cols:
            val = pivot.loc[model, col] if col in pivot.columns else np.nan
            if np.isnan(val):
                row += " & --"
            else:
                row += f" & {fmt(val, model, col)}"
        for col in relation_cols:
            val = pivot.loc[model, col] if col in pivot.columns else np.nan
            if np.isnan(val):
                row += " & --"
            else:
                row += f" & {fmt(val, model, col)}"
        row += " \\\\"
        lines.append(row)
    
    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    
    return "\n".join(lines)


# Generate per-condition table
per_condition_latex = generate_per_condition_latex_table(
    per_condition_df, CONDITION_MAP, ENTITY_CONDITIONS, RELATION_CONDITIONS
)

print("\n" + "="*80)
print("TABLE 2: PER-CONDITION ACCURACY")
print("="*80)
print(per_condition_latex)


TABLE 2: PER-CONDITION ACCURACY
\begin{tabular}{@{}lccc|ccccc@{}}
\toprule
& \multicolumn{3}{c|}{\textbf{Entity}} & \multicolumn{5}{c}{\textbf{Relation}} \\
\cmidrule(r){2-4}\cmidrule(l){5-9}
\textbf{Model} & +Obj & +Attr & +Rand & Attr & Obj & Ant & Swap & Subj \\
\midrule
CE-CLIP & 48.0 & 31.5 & 78.9 & 23.5 & 36.7 & 21.3 & 63.9 & 57.3 \\
CLIC-LAION & 49.0 & 33.1 & 82.8 & 20.5 & 33.5 & 13.7 & 22.5 & 58.4 \\
CLIC-RedCaps & 45.2 & 27.1 & 79.3 & 22.7 & 35.5 & 15.7 & 22.0 & 60.3 \\
CLIP & 41.3 & 27.1 & 72.6 & 25.5 & 37.9 & 21.0 & 24.2 & 59.4 \\
CLoVe & 43.4 & 32.1 & 80.3 & 21.8 & 32.1 & 19.9 & 25.8 & 48.6 \\
CON-CLIP & \textbf{69.1} & 48.0 & 91.9 & 45.9 & 64.4 & 34.2 & 34.0 & 72.4 \\
CS-CLIP & 68.9 & 43.6 & 92.2 & 40.1 & \textbf{66.6} & \textbf{66.8} & 64.1 & \textbf{75.7} \\
DAC-LLM & 49.1 & 34.1 & 81.5 & 31.9 & 43.5 & 24.7 & 28.4 & 61.6 \\
DAC-SAM & 46.7 & 32.1 & 78.7 & 32.2 & 43.5 & 25.1 & 27.8 & 61.2 \\
DEGLA & 56.5 & 40.5 & 88.7 & 26.5 & 38.3 & 20.9 & 41.8 & 61.4 \\
FSC-CLIP-CC3M & 

In [5]:
# =============================================================================
# TABLE 3: FULL-TRUTH TABLE (FT > HT accuracy)
# =============================================================================

def compute_full_truth_metrics(all_results, entity_conditions, relation_conditions, condition_map):
    """
    Compute Full-Truth accuracy: whether score_long_correct > score_long_incorrect.
    This measures if the model correctly prefers the full caption over the half-truth.
    
    Column mapping:
    - score_short_correct = Short-Truth (ST)
    - score_long_incorrect = Half-Truth (HT)
    - score_long_correct = Full-Truth (FT)
    """
    summary_records = []
    per_condition_records = []
    
    for model_name, df in all_results.items():
        # Filter negation
        df_filtered = df[df['condition'] != 'relation_negation'].copy()
        
        # Check if we have the necessary columns
        if 'score_long_correct' not in df_filtered.columns or 'score_long_incorrect' not in df_filtered.columns:
            print(f"  ⚠ {model_name}: Missing score_long_correct or score_long_incorrect columns")
            continue
        
        # FT-Acc = score_long_correct > score_long_incorrect (Full-Truth > Half-Truth)
        df_filtered['ft_wins'] = (df_filtered['score_long_correct'] > df_filtered['score_long_incorrect']).astype(float)
        
        # Entity FT-Acc (micro-averaged)
        ent_subset = df_filtered[df_filtered['condition'].isin(entity_conditions)]
        ent_acc = ent_subset['ft_wins'].mean() * 100 if len(ent_subset) > 0 else np.nan
        ent_n = len(ent_subset)
        
        # Relation FT-Acc (micro-averaged)
        rel_subset = df_filtered[df_filtered['condition'].isin(relation_conditions)]
        rel_acc = rel_subset['ft_wins'].mean() * 100 if len(rel_subset) > 0 else np.nan
        rel_n = len(rel_subset)
        
        # Overall FT-Acc (micro-averaged)
        all_conditions = entity_conditions + relation_conditions
        all_subset = df_filtered[df_filtered['condition'].isin(all_conditions)]
        overall_acc = all_subset['ft_wins'].mean() * 100 if len(all_subset) > 0 else np.nan
        overall_n = len(all_subset)
        
        # Gap
        gap = ent_acc - rel_acc if not np.isnan(ent_acc) and not np.isnan(rel_acc) else np.nan
        
        summary_records.append({
            'Model': model_name,
            'Entity': ent_acc,
            'Entity_n': ent_n,
            'Relation': rel_acc,
            'Relation_n': rel_n,
            'Gap': gap,
            'Overall': overall_acc,
            'Overall_n': overall_n,
        })
        
        # Per-condition FT-Acc
        for condition, (display_name, category) in condition_map.items():
            cond_subset = df_filtered[df_filtered['condition'] == condition]
            if len(cond_subset) == 0:
                continue
            acc = cond_subset['ft_wins'].mean() * 100
            per_condition_records.append({
                'model': model_name,
                'condition': condition,
                'display_name': display_name,
                'category': category,
                'accuracy': acc,
                'n': len(cond_subset),
            })
    
    if len(summary_records) == 0:
        return pd.DataFrame(), pd.DataFrame()
    
    return pd.DataFrame(summary_records).set_index('Model'), pd.DataFrame(per_condition_records)


# Compute Full-Truth metrics
print("\n" + "="*60)
print("COMPUTING FULL-TRUTH METRICS (FT > HT)")
print("="*60)

ft_summary_df, ft_per_condition_df = compute_full_truth_metrics(
    all_model_results, ENTITY_CONDITIONS, RELATION_CONDITIONS, CONDITION_MAP
)

if len(ft_summary_df) > 0:
    print("\nFull-Truth Summary (Micro-Averaged FT-Acc %):")
    print(ft_summary_df[['Entity', 'Relation', 'Gap', 'Overall']].round(1).to_string())
    
    # Generate FT summary table
    ft_summary_latex = generate_summary_latex_table(ft_summary_df)
    
    print("\n" + "="*80)
    print("TABLE 3: FULL-TRUTH SUMMARY (FT > HT)")
    print("="*80)
    print(ft_summary_latex)
    
    # Generate FT per-condition table
    if len(ft_per_condition_df) > 0:
        ft_per_condition_latex = generate_per_condition_latex_table(
            ft_per_condition_df, CONDITION_MAP, ENTITY_CONDITIONS, RELATION_CONDITIONS
        )
        
        print("\n" + "="*80)
        print("TABLE 4: FULL-TRUTH PER-CONDITION")
        print("="*80)
        print(ft_per_condition_latex)
else:
    ft_per_condition_latex = None
    ft_summary_latex = None
    print("⚠ No Full-Truth data available (missing score columns)")


COMPUTING FULL-TRUTH METRICS (FT > HT)

Full-Truth Summary (Micro-Averaged FT-Acc %):
                Entity  Relation   Gap  Overall
Model                                          
CLIP              84.9      75.8   9.1     79.3
CE-CLIP           89.3      89.0   0.4     89.1
CLIC-LAION        86.7      79.0   7.8     81.9
CLIC-RedCaps      86.5      77.8   8.7     81.2
CLoVe             87.8      84.4   3.4     85.7
CON-CLIP          85.0      69.3  15.7     75.3
CS-CLIP           90.3      91.6  -1.3     91.1
DAC-LLM           84.7      76.5   8.2     79.6
DAC-SAM           84.6      75.3   9.3     78.9
DEGLA             88.2      88.3  -0.2     88.3
FSC-CLIP-CC3M     88.1      86.2   2.0     86.9
FSC-CLIP-COCO     88.6      85.9   2.7     86.9
FSC-CLIP-LAION    87.2      86.0   1.1     86.5
LabCLIP           87.2      83.1   4.1     84.6
LA-CLIP           73.9      65.0   8.8     68.4
NegCLIP           86.8      84.1   2.7     85.1
ReadCLIP          87.9      87.3   0.6     87.6
S

In [6]:
# =============================================================================
# TABLE: HT-Acc with ST vs HT Delta (Entity and Relation separately)
# =============================================================================
# Shows accuracy and the gap between ST and HT scores for each category

def compute_ht_with_delta(all_results, entity_conditions, relation_conditions, condition_map):
    """
    Compute HT-Acc (ST > HT) and the average score delta (ST - HT) for Entity and Relation.
    
    - HT-Acc = percentage where score_short_correct > score_long_incorrect
    - Delta = average (score_short_correct - score_long_incorrect)
    """
    records = []
    
    for model_name, df in all_results.items():
        # Filter negation
        df_filtered = df[df['condition'] != 'relation_negation'].copy()
        
        # Check columns
        if 'score_short_correct' not in df_filtered.columns or 'score_long_incorrect' not in df_filtered.columns:
            continue
        
        # Compute delta (ST - HT)
        df_filtered['delta_st_ht'] = df_filtered['score_short_correct'] - df_filtered['score_long_incorrect']
        # HT-Acc = ST > HT
        df_filtered['st_wins'] = (df_filtered['score_short_correct'] > df_filtered['score_long_incorrect']).astype(float)
        
        # Entity metrics
        ent_subset = df_filtered[df_filtered['condition'].isin(entity_conditions)]
        ent_acc = ent_subset['st_wins'].mean() * 100 if len(ent_subset) > 0 else np.nan
        ent_delta = ent_subset['delta_st_ht'].mean() if len(ent_subset) > 0 else np.nan
        ent_n = len(ent_subset)
        
        # Relation metrics
        rel_subset = df_filtered[df_filtered['condition'].isin(relation_conditions)]
        rel_acc = rel_subset['st_wins'].mean() * 100 if len(rel_subset) > 0 else np.nan
        rel_delta = rel_subset['delta_st_ht'].mean() if len(rel_subset) > 0 else np.nan
        rel_n = len(rel_subset)
        
        # Overall
        all_conditions = entity_conditions + relation_conditions
        all_subset = df_filtered[df_filtered['condition'].isin(all_conditions)]
        overall_acc = all_subset['st_wins'].mean() * 100 if len(all_subset) > 0 else np.nan
        overall_delta = all_subset['delta_st_ht'].mean() if len(all_subset) > 0 else np.nan
        
        records.append({
            'Model': model_name,
            'Ent_Acc': ent_acc,
            'Ent_Delta': ent_delta,
            'Rel_Acc': rel_acc,
            'Rel_Delta': rel_delta,
            'Overall_Acc': overall_acc,
            'Overall_Delta': overall_delta,
        })
    
    if len(records) == 0:
        return pd.DataFrame()
    
    return pd.DataFrame(records).set_index('Model')


def generate_ht_delta_latex_table(df, model_order=None):
    """
    Generate LaTeX table showing HT-Acc with ST-HT delta for Entity and Relation.
    Columns: Model | Ent Acc | Ent Δ | Rel Acc | Rel Δ | Overall Acc | Overall Δ
    """
    if model_order is None:
        model_order = df.sort_values('Overall_Acc', ascending=False).index.tolist()
    else:
        model_order = [m for m in model_order if m in df.index]
    
    # Find best per column
    best_ent_acc = df['Ent_Acc'].idxmax()
    best_rel_acc = df['Rel_Acc'].idxmax()
    best_overall_acc = df['Overall_Acc'].idxmax()
    # For delta, higher is better (more margin)
    best_ent_delta = df['Ent_Delta'].idxmax()
    best_rel_delta = df['Rel_Delta'].idxmax()
    best_overall_delta = df['Overall_Delta'].idxmax()
    
    def fmt_acc(val, model, best):
        s = f"{val:.1f}"
        if model == best:
            s = f"\\textbf{{{s}}}"
        return s
    
    def fmt_delta(val, model, best):
        if val > 0:
            s = f"+{val:.3f}"
        else:
            s = f"{val:.3f}"
        if model == best:
            s = f"\\textbf{{{s}}}"
        return s
    
    lines = []
    lines.append(r"\begin{tabular}{@{}l|cc|cc|cc@{}}")
    lines.append(r"\toprule")
    lines.append(r"& \multicolumn{2}{c|}{\textbf{Entity}} & \multicolumn{2}{c|}{\textbf{Relation}} & \multicolumn{2}{c}{\textbf{Overall}} \\")
    lines.append(r"\cmidrule(r){2-3}\cmidrule(lr){4-5}\cmidrule(l){6-7}")
    lines.append(r"\textbf{Model} & Acc & $\Delta$ & Acc & $\Delta$ & Acc & $\Delta$ \\")
    lines.append(r"\midrule")
    
    for model in model_order:
        row = model.replace("_", r"\_")
        row += f" & {fmt_acc(df.loc[model, 'Ent_Acc'], model, best_ent_acc)}"
        row += f" & {fmt_delta(df.loc[model, 'Ent_Delta'], model, best_ent_delta)}"
        row += f" & {fmt_acc(df.loc[model, 'Rel_Acc'], model, best_rel_acc)}"
        row += f" & {fmt_delta(df.loc[model, 'Rel_Delta'], model, best_rel_delta)}"
        row += f" & {fmt_acc(df.loc[model, 'Overall_Acc'], model, best_overall_acc)}"
        row += f" & {fmt_delta(df.loc[model, 'Overall_Delta'], model, best_overall_delta)} \\\\"
        lines.append(row)
    
    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    
    return "\n".join(lines)


# Compute HT with delta
print("\n" + "="*60)
print("COMPUTING HT-Acc WITH ST-HT DELTA")
print("="*60)

ht_delta_df = compute_ht_with_delta(
    all_model_results, ENTITY_CONDITIONS, RELATION_CONDITIONS, CONDITION_MAP
)

if len(ht_delta_df) > 0:
    print("\nHT-Acc with Delta (ST - HT):")
    print(ht_delta_df.round(3).to_string())
    
    # Generate table
    ht_delta_latex = generate_ht_delta_latex_table(ht_delta_df)
    
    print("\n" + "="*80)
    print("TABLE: HT-Acc WITH ST-HT DELTA")
    print("="*80)
    print(ht_delta_latex)
else:
    ht_delta_latex = None
    print("⚠ No data available")


COMPUTING HT-Acc WITH ST-HT DELTA

HT-Acc with Delta (ST - HT):
                Ent_Acc  Ent_Delta  Rel_Acc  Rel_Delta  Overall_Acc  Overall_Delta
Model                                                                             
CLIP             52.942     -0.002   32.899     -0.017       40.568         -0.011
CE-CLIP          59.081      0.002   42.949     -0.006       49.121         -0.003
CLIC-LAION       61.462      0.005   29.106     -0.021       41.485         -0.011
CLIC-RedCaps     57.504      0.002   30.441     -0.018       40.795         -0.010
CLoVe            57.907      0.001   29.551     -0.021       40.400         -0.013
CON-CLIP         75.981      0.020   48.825     -0.002       59.215          0.007
CS-CLIP          75.420      0.021   65.467      0.014       69.275          0.017
DAC-LLM          61.103      0.004   37.143     -0.014       46.310         -0.007
DAC-SAM          58.581      0.002   37.018     -0.013       45.268         -0.008
DEGLA            68.28

In [7]:
# =============================================================================
# TABLE 5: CONSISTENCY TABLE (FT > ST)
# =============================================================================
# Tests whether adding more relevant correct information increases the score
# i.e., Full-Truth (long correct) > Short-Truth (short correct)

def compute_consistency_metrics(all_results, entity_conditions, relation_conditions, condition_map):
    """
    Compute Consistency: whether score_long_correct > score_short_correct.
    This measures if adding more correct information increases the score.
    
    Column mapping:
    - score_short_correct = Short-Truth (ST)
    - score_long_correct = Full-Truth (FT)
    
    Consistency = FT > ST (more complete caption preferred over shorter one)
    """
    summary_records = []
    per_condition_records = []
    
    for model_name, df in all_results.items():
        # Filter negation
        df_filtered = df[df['condition'] != 'relation_negation'].copy()
        
        # Check if we have the necessary columns
        if 'score_long_correct' not in df_filtered.columns or 'score_short_correct' not in df_filtered.columns:
            print(f"  ⚠ {model_name}: Missing score_long_correct or score_short_correct columns")
            continue
        
        # Consistency = score_long_correct > score_short_correct (Full-Truth > Short-Truth)
        df_filtered['consistent'] = (df_filtered['score_long_correct'] > df_filtered['score_short_correct']).astype(float)
        
        # Entity Consistency (micro-averaged)
        ent_subset = df_filtered[df_filtered['condition'].isin(entity_conditions)]
        ent_acc = ent_subset['consistent'].mean() * 100 if len(ent_subset) > 0 else np.nan
        ent_n = len(ent_subset)
        
        # Relation Consistency (micro-averaged)
        rel_subset = df_filtered[df_filtered['condition'].isin(relation_conditions)]
        rel_acc = rel_subset['consistent'].mean() * 100 if len(rel_subset) > 0 else np.nan
        rel_n = len(rel_subset)
        
        # Overall Consistency (micro-averaged)
        all_conditions = entity_conditions + relation_conditions
        all_subset = df_filtered[df_filtered['condition'].isin(all_conditions)]
        overall_acc = all_subset['consistent'].mean() * 100 if len(all_subset) > 0 else np.nan
        overall_n = len(all_subset)
        
        # Gap
        gap = ent_acc - rel_acc if not np.isnan(ent_acc) and not np.isnan(rel_acc) else np.nan
        
        summary_records.append({
            'Model': model_name,
            'Entity': ent_acc,
            'Entity_n': ent_n,
            'Relation': rel_acc,
            'Relation_n': rel_n,
            'Gap': gap,
            'Overall': overall_acc,
            'Overall_n': overall_n,
        })
        
        # Per-condition Consistency
        for condition, (display_name, category) in condition_map.items():
            cond_subset = df_filtered[df_filtered['condition'] == condition]
            if len(cond_subset) == 0:
                continue
            acc = cond_subset['consistent'].mean() * 100
            per_condition_records.append({
                'model': model_name,
                'condition': condition,
                'display_name': display_name,
                'category': category,
                'accuracy': acc,
                'n': len(cond_subset),
            })
    
    if len(summary_records) == 0:
        return pd.DataFrame(), pd.DataFrame()
    
    return pd.DataFrame(summary_records).set_index('Model'), pd.DataFrame(per_condition_records)


# Compute Consistency metrics
print("\n" + "="*60)
print("COMPUTING CONSISTENCY METRICS (FT > ST)")
print("="*60)

cons_summary_df, cons_per_condition_df = compute_consistency_metrics(
    all_model_results, ENTITY_CONDITIONS, RELATION_CONDITIONS, CONDITION_MAP
)

if len(cons_summary_df) > 0:
    print("\nConsistency Summary (Micro-Averaged FT > ST %):")
    print(cons_summary_df[['Entity', 'Relation', 'Gap', 'Overall']].round(1).to_string())
    
    # Generate Consistency summary table
    cons_summary_latex = generate_summary_latex_table(cons_summary_df)
    
    print("\n" + "="*80)
    print("TABLE 5: CONSISTENCY SUMMARY (FT > ST)")
    print("="*80)
    print(cons_summary_latex)
    
    # Generate Consistency per-condition table
    if len(cons_per_condition_df) > 0:
        cons_per_condition_latex = generate_per_condition_latex_table(
            cons_per_condition_df, CONDITION_MAP, ENTITY_CONDITIONS, RELATION_CONDITIONS
        )
        
        print("\n" + "="*80)
        print("TABLE 6: CONSISTENCY PER-CONDITION")
        print("="*80)
        print(cons_per_condition_latex)
else:
    cons_per_condition_latex = None
    cons_summary_latex = None
    print("⚠ No Consistency data available (missing score columns)")


COMPUTING CONSISTENCY METRICS (FT > ST)

Consistency Summary (Micro-Averaged FT > ST %):
                Entity  Relation   Gap  Overall
Model                                          
CLIP              83.9      82.1   1.8     82.8
CE-CLIP           85.3      90.3  -5.0     88.4
CLIC-LAION        82.6      89.6  -6.9     86.9
CLIC-RedCaps      84.9      87.4  -2.5     86.5
CLoVe             84.7      90.7  -6.0     88.4
CON-CLIP          67.1      67.5  -0.4     67.4
CS-CLIP           80.2      82.5  -2.3     81.6
DAC-LLM           78.8      80.6  -1.8     79.9
DAC-SAM           79.8      79.4   0.3     79.6
DEGLA             78.0      90.0 -12.0     85.4
FSC-CLIP-CC3M     65.9      67.3  -1.4     66.8
FSC-CLIP-COCO     75.2      80.2  -4.9     78.3
FSC-CLIP-LAION    73.7      76.7  -2.9     75.6
LabCLIP           77.8      83.8  -6.1     81.5
LA-CLIP           61.9      65.8  -3.8     64.3
NegCLIP           74.9      77.4  -2.4     76.4
ReadCLIP          75.7      83.5  -7.8     80.

In [8]:
# =============================================================================
# SAVE ALL TABLES TO FILES
# =============================================================================

import os
os.makedirs("../paper_figures", exist_ok=True)

# Save HT Summary Table
with open("../paper_figures/ht_summary_table.tex", 'w') as f:
    f.write("% Half-Truth Summary: Entity vs Relation (Micro-Averaged)\n")
    f.write(summary_latex)
print("✓ Saved: ../paper_figures/ht_summary_table.tex")

# Save HT Per-Condition Table
with open("../paper_figures/ht_per_condition_table.tex", 'w') as f:
    f.write("% Half-Truth Per-Condition Accuracy\n")
    f.write(per_condition_latex)
print("✓ Saved: ../paper_figures/ht_per_condition_table.tex")

# Save FT Tables if available
if len(ft_summary_df) > 0 and ft_summary_latex is not None:
    with open("../paper_figures/ft_summary_table.tex", 'w') as f:
        f.write("% Full-Truth Summary: FT > HT (Micro-Averaged)\n")
        f.write(ft_summary_latex)
    print("✓ Saved: ../paper_figures/ft_summary_table.tex")
    
    if len(ft_per_condition_df) > 0 and ft_per_condition_latex is not None:
        with open("../paper_figures/ft_per_condition_table.tex", 'w') as f:
            f.write("% Full-Truth Per-Condition Accuracy\n")
            f.write(ft_per_condition_latex)
        print("✓ Saved: ../paper_figures/ft_per_condition_table.tex")

# Save Consistency Tables if available
if len(cons_summary_df) > 0 and cons_summary_latex is not None:
    with open("../paper_figures/consistency_summary_table.tex", 'w') as f:
        f.write("% Consistency Summary: FT > ST (Micro-Averaged)\n")
        f.write(cons_summary_latex)
    print("✓ Saved: ../paper_figures/consistency_summary_table.tex")
    
    if len(cons_per_condition_df) > 0 and cons_per_condition_latex is not None:
        with open("../paper_figures/consistency_per_condition_table.tex", 'w') as f:
            f.write("% Consistency Per-Condition Accuracy\n")
            f.write(cons_per_condition_latex)
        print("✓ Saved: ../paper_figures/consistency_per_condition_table.tex")

# Save HT Delta Table if available
if len(ht_delta_df) > 0 and ht_delta_latex is not None:
    with open("../paper_figures/ht_delta_table.tex", 'w') as f:
        f.write("% HT-Acc with ST-HT Delta (score gap) for Entity and Relation\n")
        f.write(ht_delta_latex)
    print("✓ Saved: ../paper_figures/ht_delta_table.tex")

print("\n✓ All tables saved!")

✓ Saved: ../paper_figures/ht_summary_table.tex
✓ Saved: ../paper_figures/ht_per_condition_table.tex
✓ Saved: ../paper_figures/ft_summary_table.tex
✓ Saved: ../paper_figures/ft_per_condition_table.tex
✓ Saved: ../paper_figures/consistency_summary_table.tex
✓ Saved: ../paper_figures/consistency_per_condition_table.tex
✓ Saved: ../paper_figures/ht_delta_table.tex

✓ All tables saved!
